In [ ]:
# 必要に応じて追加のライブラリをインストール
!pip install torch torchvision torchaudio
!pip install pandas numpy matplotlib tqdm opencv-python

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    
    # Google Driveをマウント
    drive.mount('/content/drive')
    
    # プロジェクトディレクトリに移動
    PROJECT_ROOT = '/content/drive/MyDrive/Visuable_for_you_tabletennis'
    os.chdir(PROJECT_ROOT)
    
    # notebooksディレクトリをパスに追加
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts/notebooks'))
    
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True
    
except ImportError:
    # ローカル環境の場合
    IN_COLAB = False
    # notebooksディレクトリ（このノートブックの場所）をパスに追加
    notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
import os
import sys

# プロジェクトのパスを指定（例: /content/drive/MyDrive/Visuable_for_you_tabletennis）
PROJECT_PATH = '/content/drive/MyDrive/Visuable_for_you_tabletennis'

if not os.path.exists(PROJECT_PATH):
    print(f"プロジェクトが見つかりません: {PROJECT_PATH}")
    print("パスを確認してください")
else:
    print(f"プロジェクトを使用: {PROJECT_PATH}")

# プロジェクトパスをPythonのパスに追加
sys.path.insert(0, PROJECT_PATH)

# 作業ディレクトリを変更
os.chdir(PROJECT_PATH)
print(f"作業ディレクトリ: {os.getcwd()}")

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
import json
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML

# プロジェクトのモジュールをインポート
from src.models.play_classifier import PlayClassifierLSTM, PlayClassifierCNNLSTM
from src.dataset.dataset import PoseSequenceDataset

print("インポート完了")
print(f"PyTorchバージョン: {torch.__version__}")
print(f"OpenCVバージョン: {cv2.__version__}")
print(f"CUDAが利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# パス設定（ここを変更してください）
# ============================================================

# 学習済みモデルのパス
MODEL_PATH = '/content/drive/MyDrive/trained_models/play_classifier/20260129_XXXXXX/best_model.pth'
CONFIG_PATH = '/content/drive/MyDrive/trained_models/play_classifier/20260129_XXXXXX/config.json'

# 予測対象の動画データ
VIDEO_NAME = 'sample_video_01_02'  # 予測したい動画名
DATA_DIR = f'/content/drive/MyDrive/Visuable_for_you_tabletennis/data/detect/{VIDEO_NAME}'

# 入力ファイル
POSE_CSV_PATH = f"{DATA_DIR}/original_pose_data.csv"  # 骨格データCSV
VIDEO_PATH = f"/content/drive/MyDrive/Visuable_for_you_tabletennis/data/raw/{VIDEO_NAME}.MOV"  # 元動画

# 出力ディレクトリ
OUTPUT_DIR = f'/content/drive/MyDrive/predictions/{VIDEO_NAME}'

# デバイス設定
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 予測の閾値
THRESHOLD = 0.5

# ============================================================

# 出力ディレクトリ作成
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ファイル存在確認
print("="*60)
print("ファイルの確認")
print("="*60)
print(f"モデル: {'✓' if os.path.exists(MODEL_PATH) else '✗'} {MODEL_PATH}")
print(f"設定: {'✓' if os.path.exists(CONFIG_PATH) else '✗'} {CONFIG_PATH}")
print(f"骨格CSV: {'✓' if os.path.exists(POSE_CSV_PATH) else '✗'} {POSE_CSV_PATH}")
print(f"動画: {'✓' if os.path.exists(VIDEO_PATH) else '✗'} {VIDEO_PATH}")
print(f"出力先: {OUTPUT_DIR}")
print(f"デバイス: {DEVICE}")
print("="*60)

In [ ]:
# 設定ファイルを読み込み
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, 'r') as f:
        config = json.load(f)
    print("設定ファイルを読み込みました:")
    for key, value in config.items():
        print(f"  {key}: {value}")
else:
    # デフォルト設定
    print("警告: 設定ファイルが見つかりません。デフォルト設定を使用します。")
    config = {
        'model_type': 'lstm',
        'hidden_size': 128,
        'num_layers': 2,
        'dropout': 0.3,
        'no_attention': False,
        'sequence_length': 30
    }

# モデル作成
print("\nモデルを作成中...")
if config['model_type'] == 'lstm':
    model = PlayClassifierLSTM(
        input_size=34,
        hidden_size=config['hidden_size'],
        num_layers=config['num_layers'],
        dropout=config['dropout'],
        use_attention=not config.get('no_attention', False)
    )
else:  # cnn_lstm
    model = PlayClassifierCNNLSTM(
        input_size=34,
        hidden_size=config['hidden_size'],
        num_layers=config['num_layers'],
        dropout=config['dropout']
    )

# 重み読み込み
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

print(f"モデル読み込み完了: {MODEL_PATH}")
print(f"エポック: {checkpoint.get('epoch', 'N/A')}")
print(f"Best Val F1: {checkpoint.get('best_val_f1', 'N/A')}")

In [ ]:
# データセット作成
sequence_length = config.get('sequence_length', 30)
dataset = PoseSequenceDataset(
    csv_path=POSE_CSV_PATH,
    label_path=None,  # ラベルなし（予測のみ）
    sequence_length=sequence_length,
    stride=1  # 全フレームを予測するためstride=1
)

print(f"予測開始:")
print(f"  入力CSV: {POSE_CSV_PATH}")
print(f"  総フレーム数: {len(dataset.data_df)}")
print(f"  シーケンス数: {len(dataset)}")
print(f"  シーケンス長: {sequence_length}フレーム")

# 各フレームの予測確率を集計
frame_probs = {}  # {frame_num: [probs]}

with torch.no_grad():
    for features, _, metadata in tqdm(dataset, desc="予測中"):
        # バッチ次元を追加
        features = features.unsqueeze(0).to(DEVICE)
        
        # 予測
        outputs = model(features)  # (1, seq, 1)
        probs = outputs.squeeze().cpu().numpy()  # (seq,)
        
        # フレームごとに確率を記録
        start_frame = metadata['start_frame']
        for i, prob in enumerate(probs):
            frame_num = start_frame + i
            if frame_num not in frame_probs:
                frame_probs[frame_num] = []
            frame_probs[frame_num].append(prob)

# 各フレームの確率を平均
predictions = []
for frame_num in sorted(frame_probs.keys()):
    avg_prob = np.mean(frame_probs[frame_num])
    prediction = 1 if avg_prob >= THRESHOLD else 0
    predictions.append({
        'frame': frame_num,
        'probability': avg_prob,
        'prediction': prediction,
        'num_predictions': len(frame_probs[frame_num])
    })

result_df = pd.DataFrame(predictions)

# 統計
num_play_frames = np.sum(result_df['prediction'] == 1)
play_ratio = num_play_frames / len(result_df) * 100
print(f"\n予測結果:")
print(f"  プレー中フレーム: {num_play_frames} / {len(result_df)} ({play_ratio:.1f}%)")

# CSV保存
output_csv_path = os.path.join(OUTPUT_DIR, 'predictions.csv')
result_df.to_csv(output_csv_path, index=False)
print(f"  予測結果保存: {output_csv_path}")

# 先頭5件を表示
print("\n予測結果（先頭5件）:")
display(result_df.head())

In [ ]:
def extract_scenes(result_df, min_duration=10):
    """
    連続したプレー区間を抽出
    
    Args:
        result_df: 予測結果のDataFrame
        min_duration: 最小シーン長（フレーム数）
    
    Returns:
        [(start_frame, end_frame), ...] のリスト
    """
    scenes = []
    in_scene = False
    scene_start = None
    
    for _, row in result_df.iterrows():
        frame = row['frame']
        pred = row['prediction']
        
        if pred == 1:  # プレー中
            if not in_scene:
                scene_start = frame
                in_scene = True
        else:  # プレー外
            if in_scene:
                # シーン終了
                if frame - scene_start >= min_duration:
                    scenes.append((scene_start, frame - 1))
                in_scene = False
    
    # 最後のシーン
    if in_scene and result_df['frame'].iloc[-1] - scene_start >= min_duration:
        scenes.append((scene_start, result_df['frame'].iloc[-1]))
    
    return scenes

# シーン検出
scenes = extract_scenes(result_df, min_duration=10)
print(f"検出シーン数: {len(scenes)}")

# シーン情報を表示（最初の10シーン）
for i, (start, end) in enumerate(scenes[:10], 1):
    duration = (end - start) / 30.0  # 30fps想定
    print(f"  シーン{i}: フレーム {start}-{end} ({duration:.1f}秒)")

# シーン情報をCSV保存
scenes_df = pd.DataFrame(scenes, columns=['start_frame', 'end_frame'])
scenes_df['duration_sec'] = (scenes_df['end_frame'] - scenes_df['start_frame']) / 30.0
scenes_csv_path = os.path.join(OUTPUT_DIR, 'scenes.csv')
scenes_df.to_csv(scenes_csv_path, index=False)
print(f"\nシーン情報保存: {scenes_csv_path}")

In [ ]:
# 確率のプロット
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# 上段: プレー中の確率
axes[0].plot(result_df['frame'], result_df['probability'], linewidth=0.8, alpha=0.7, color='blue')
axes[0].axhline(y=THRESHOLD, color='r', linestyle='--', linewidth=2, label=f'閾値 ({THRESHOLD})')
axes[0].fill_between(
    result_df['frame'],
    result_df['probability'],
    THRESHOLD,
    where=(result_df['probability'] >= THRESHOLD),
    alpha=0.3,
    color='green',
    label='プレー中と判定'
)
axes[0].set_ylabel('プレー中の確率', fontsize=12)
axes[0].set_title(f'プレー検知結果: {VIDEO_NAME}', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

# 下段: 予測結果（0/1）
axes[1].fill_between(
    result_df['frame'],
    result_df['prediction'],
    alpha=0.6,
    color='green',
    label='プレー中'
)
axes[1].set_ylabel('判定結果', fontsize=12)
axes[1].set_xlabel('フレーム番号', fontsize=12)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['プレー外', 'プレー中'])
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([-0.1, 1.1])

plt.tight_layout()

# グラフを保存
graph_path = os.path.join(OUTPUT_DIR, 'prediction_graph.png')
plt.savefig(graph_path, dpi=150, bbox_inches='tight')
print(f"グラフを保存: {graph_path}")

plt.show()

In [ ]:
# 動画の存在確認
if not os.path.exists(VIDEO_PATH):
    print(f"警告: 動画が見つかりません: {VIDEO_PATH}")
    print("動画出力をスキップします。")
else:
    print(f"動画処理中: {VIDEO_PATH}")
    
    # 動画読み込み
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise ValueError(f"動画を開けません: {VIDEO_PATH}")
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"  解像度: {width}x{height}")
    print(f"  FPS: {fps}")
    print(f"  総フレーム数: {total_frames}")
    
    # 出力動画設定
    output_video_path = os.path.join(OUTPUT_DIR, f'{VIDEO_NAME}_predicted.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    frame_idx = 0
    pbar = tqdm(total=total_frames, desc="動画書き込み")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # 予測結果を取得
        if frame_idx < len(result_df):
            prob_play = result_df.iloc[frame_idx]['probability']
            prob_non_play = 1 - prob_play
            pred = result_df.iloc[frame_idx]['prediction']
            
            # 背景（半透明の黒）
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (350, 110), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
            
            # ステータス表示
            status_text = "Playing" if pred == 1 else "Not Playing"
            status_color = (0, 255, 0) if pred == 1 else (128, 128, 128)
            cv2.putText(frame, status_text, (20, 40),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, status_color, 2)
            
            # プレー中の確率
            play_text = f"Play:     {prob_play:.1%}"
            play_color = (0, 255, 0)
            cv2.putText(frame, play_text, (20, 70),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, play_color, 2)
            
            # プレー外の確率
            non_play_text = f"Non-play: {prob_non_play:.1%}"
            non_play_color = (128, 128, 128)
            cv2.putText(frame, non_play_text, (20, 95),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, non_play_color, 2)
        
        out.write(frame)
        frame_idx += 1
        pbar.update(1)
    
    pbar.close()
    cap.release()
    out.release()
    
    print(f"\n出力動画保存: {output_video_path}")
    print(f"ファイルサイズ: {os.path.getsize(output_video_path) / (1024*1024):.1f} MB")

In [ ]:
# 出力動画を表示
if os.path.exists(output_video_path):
    print("生成された動画:")
    display(Video(output_video_path, width=800))
else:
    print("動画が見つかりません")

In [ ]:
print("="*60)
print("予測結果サマリー")
print("="*60)
print(f"動画名: {VIDEO_NAME}")
print(f"総フレーム数: {len(result_df)}")
print(f"プレー中フレーム: {num_play_frames} ({play_ratio:.1f}%)")
print(f"プレー外フレーム: {len(result_df) - num_play_frames} ({100-play_ratio:.1f}%)")
print(f"検出シーン数: {len(scenes)}")
print(f"\n平均確率:")
print(f"  全体: {result_df['probability'].mean():.3f}")
print(f"  プレー中と判定されたフレーム: {result_df[result_df['prediction']==1]['probability'].mean():.3f}")
print(f"  プレー外と判定されたフレーム: {result_df[result_df['prediction']==0]['probability'].mean():.3f}")
print(f"\n出力ファイル:")
print(f"  予測結果CSV: {output_csv_path}")
print(f"  シーン情報CSV: {scenes_csv_path}")
print(f"  可視化グラフ: {graph_path}")
if os.path.exists(output_video_path):
    print(f"  予測動画: {output_video_path}")
print("="*60)